In [ ]:
!pip install tensorflow==2.12.0
!pip install mtcnn
!pip install dlib imutils

In [ ]:
from google.colab import drive
import os
from PIL import Image
import cv2
import shutil
import torch
from IPython.display import display
from mtcnn import MTCNN
from mtcnn.utils.images import load_image
import matplotlib.pyplot as plt
import dlib
import numpy as np
from imutils.face_utils import FaceAligner
from imutils.face_utils import rect_to_bb

In [ ]:
# Acá se montan todos los archivos del drive, es necesario que le den las credenciales de la U
drive.mount('/content/drive')


In [ ]:
from mtcnn import MTCNN
import cv2
import matplotlib.pyplot as plt

detector_mtcnn = MTCNN()

def recortar_cara(imagen):
    # Convertir la imagen a RGB para MTCNN
    imagen_corregida = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

    # Detección de rostros
    resultados = detector_mtcnn.detect_faces(imagen_corregida)

    if len(resultados) == 0:
        print("No se detectó ningún rostro en la imagen.")
        plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
        plt.show()
        return imagen, None, None  # Devolver imagen original y None para confidence y keypoints

    # Obtener el primer rostro detectado
    result = resultados[0]
    x, y, w, h = result['box']
    confidence = result['confidence']
    keypoints = result['keypoints']

    # Recortar la imagen
    imagen_recortada = imagen[y:y+h, x:x+w]


    return imagen_recortada, confidence, keypoints

In [ ]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

In [ ]:
SHAPE_PREDICTOR_PATH = "shape_predictor_68_face_landmarks.dat"
OUTPUT_SIZE = (224, 224)

# Inicializar el detector de rostros y el predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(SHAPE_PREDICTOR_PATH)

def align_and_resize_face(image: np.ndarray) -> np.ndarray:
    if image is None or not isinstance(image, np.ndarray):
        print("Imagen inválida. Devolviendo imagen negra redimensionada.")
        return cv2.resize(np.zeros((100, 100, 3), dtype=np.uint8), OUTPUT_SIZE)

    original_image = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = detector(gray, 1)

    if len(faces) == 0:
        print("No se detectó ningún rostro. Redimensionando imagen original.")
        return cv2.resize(original_image, OUTPUT_SIZE)

    face = faces[0]
    landmarks = predictor(gray, face)
    landmarks_np = np.array([[p.x, p.y] for p in landmarks.parts()], dtype=np.float32)

    left_eye = np.mean(landmarks_np[36:42], axis=0)
    right_eye = np.mean(landmarks_np[42:48], axis=0)

    # Calcular ángulo
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    angle = np.degrees(np.arctan2(dy, dx))

    # Calcular centro entre los ojos
    eyes_center = ((left_eye[0] + right_eye[0]) / 2,
                   (left_eye[1] + right_eye[1]) / 2)

    # Distancia entre ojos en la imagen original
    dist = np.sqrt(dx**2 + dy**2)

    # Distancia deseada entre ojos
    desired_eye_dist = OUTPUT_SIZE[0] * (0.68 - 0.32)
    scale = desired_eye_dist / dist

    # Obtener matriz de rotación + escala
    M = cv2.getRotationMatrix2D(eyes_center, angle, scale)

    # Mover los ojos a la posición deseada
    tx = OUTPUT_SIZE[0] * 0.5 - eyes_center[0]
    ty = OUTPUT_SIZE[1] * 0.4 - eyes_center[1]
    M[0, 2] += tx
    M[1, 2] += ty

    aligned = cv2.warpAffine(image, M, OUTPUT_SIZE, flags=cv2.INTER_LINEAR)



    return aligned

In [ ]:
import matplotlib.pyplot as plt

def procesar_imagen(imagen):
    #plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    #plt.show()

    # Aplicar el pipeline
    imagen_procesada, confidence, keypoints = recortar_cara(imagen)
    imagen_procesada = align_and_resize_face(imagen_procesada)
    imagen_procesada, confidence, keypoints = recortar_cara(imagen_procesada)

    return imagen_procesada, confidence, keypoints

In [ ]:
import pandas as pd

# Specify the path to the CSV file in your shared drive
csv_path = '/content/drive/Shared drives/ClasifEye/datos.csv'
shared_drive_path = '/content/drive/Shareddrives/ClasifEye'
input_folder = '/content/drive/Shareddrives/ClasifEye/Imagenes'
output_folder = '/content/drive/Shareddrives/ClasifEye/Dataset Preprocesado'


# leer dataframe
def procesar_datos(inicio, fin, df):
    for idx in range(inicio, fin):
        print(f"Procesando imagen {idx}")
        row = df.iloc[idx]
        img_path = os.path.join(input_folder, row['archivo'])

        if not os.path.exists(img_path):
            print(f"Imagen no encontrada: {img_path}")
            continue

        imagen = cv2.imread(img_path)

        try:
            imagen_proc, confidence, keypoints = procesar_imagen(imagen)

            # Guardar imagen procesada (opcional)
            output_path = os.path.join(output_folder, f"{row['archivo']}")
            cv2.imwrite(output_path, imagen_proc)

            # Guardar resultados en DataFrame
            df.at[idx, 'mtcnn_confidence'] = confidence

            df.at[idx, 'nose_x']         = keypoints['nose'][0]
            df.at[idx, 'nose_y']         = keypoints['nose'][1]

            df.at[idx, 'mouth_right_x']  = keypoints['mouth_right'][0]
            df.at[idx, 'mouth_right_y']  = keypoints['mouth_right'][1]

            df.at[idx, 'right_eye_x']    = keypoints['right_eye'][0]
            df.at[idx, 'right_eye_y']    = keypoints['right_eye'][1]

            df.at[idx, 'left_eye_x']     = keypoints['left_eye'][0]
            df.at[idx, 'left_eye_y']     = keypoints['left_eye'][1]

            df.at[idx, 'mouth_left_x']   = keypoints['mouth_left'][0]
            df.at[idx, 'mouth_left_y']   = keypoints['mouth_left'][1]

        except Exception as e:
            print(f"Error procesando imagen {img_path}: {e}")

    return df

In [ ]:
# Leer el CSV
df = pd.read_csv(csv_path)

# Procesar del índice
df = procesar_datos(1, 6000, df)

output_csv_path = '/content/drive/Shared drives/ClasifEye/datos.csv'

# Save the DataFrame to CSV. Set index=False to avoid writing the DataFrame index as a column.
df.to_csv(output_csv_path, index=False)

print(f"DataFrame saved successfully to {output_csv_path}")